Exercise 5 — Customer Support Chat Log to Escalation Report Pipeline
Build a 3-step pipeline. Step 1 normalizes a messy support chat log into a structured summary (issue, customer tone, resolution attempted). Step 2 checks the structured summary for missing information needed to escalate the issue (e.g. account ID, error code) and flags any gaps. Step 3 generates a prioritized escalation report for the support lead based on the structured, gap-checked version.

Input: chat_log_text

Output: the structured summary from Step 1, the flagged gaps from Step 2, and the escalation report from Step 3

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: issue, customer_tone, resolution_attempted)

Step 2 — Template: ChatPromptTemplate (system persona: "escalation-readiness reviewer")  |  Parser: PydanticOutputParser (fields: missing_fields: list, is_ready_to_escalate: bool)

Step 3 — Template: ChatPromptTemplate (system persona: "support operations lead")  |  Parser: StrOutputParser

In [1]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import (
    PydanticOutputParser,
    StrOutputParser
)

In [2]:
class SupportSummary(BaseModel):
    issue: str = Field(
        description="The main problem reported by the customer"
    )

    customer_tone: str = Field(
        description="The emotional tone expressed by the customer"
    )

    resolution_attempted: str = Field(
        description="Troubleshooting or resolution steps already attempted"
    )

In [3]:
ex5_step1_parser = PydanticOutputParser(
    pydantic_object=SupportSummary
)

In [4]:
ex5_step1_prompt = PromptTemplate(
    template="""
You are extracting a structured summary from a customer
support chat log.

Extract:
- issue
- customer_tone
- resolution_attempted

Rules:
- Use only information present in the chat log.
- Do not invent error codes, account details, or actions.
- Preserve the meaning of the customer's complaint.
- If information is missing, explicitly state that it
  was not provided.
- Keep each field concise.

Customer support chat log:
{chat_log_text}

{format_instructions}
""",
    input_variables=["chat_log_text"],
    partial_variables={
        "format_instructions":
            ex5_step1_parser.get_format_instructions()
    }
)

In [5]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [6]:
ex5_step1_chain = (
    ex5_step1_prompt
    | llm
    | ex5_step1_parser
)

In [7]:
chat_log_text = """
Customer: I've been trying to check out since this morning,
but I keep getting error 502.

Support: Have you tried refreshing the page?

Customer: Yes, I refreshed the page and logged out and back
in, but nothing works. This is really frustrating!

Support: Can you provide your account ID?

Customer: I don't have it with me.
"""

support_summary = ex5_step1_chain.invoke({
    "chat_log_text": chat_log_text
})

print(support_summary.model_dump_json(indent=2))

{
  "issue": "Error 502 encountered during checkout",
  "customer_tone": "Frustrated",
  "resolution_attempted": "Customer refreshed the page and logged out and back in; support suggested refreshing and asked for account ID"
}


## Step 2 — Template: ChatPromptTemplate (system persona: "escalation-readiness reviewer")  |  Parser: PydanticOutputParser (fields: missing_fields: list, is_ready_to_escalate: bool)



In [8]:
class EscalationReadiness(BaseModel):
    missing_fields: list[str] = Field(
        description="Information missing for escalation, such as account ID or error code"
    )

    is_ready_to_escalate: bool = Field(
        description="Whether the case contains the required information for escalation"
    )

In [9]:
ex5_step2_parser = PydanticOutputParser(
    pydantic_object=EscalationReadiness
)

In [10]:
ex5_step2_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an escalation-readiness reviewer.

Review the structured customer support summary and identify
information required for escalation that is missing.

Check for:
- Account ID or account identifier
- Error code, if relevant to the issue
- A clear description of the issue
- Resolution or troubleshooting steps already attempted

Rules:
- Use only the supplied structured summary.
- Do not invent missing information.
- Include missing information in missing_fields.
- If the issue is clearly described but an optional detail
  is missing, do not automatically block escalation.
- Set is_ready_to_escalate to True only when the
  information required by the escalation process is present.
- For this exercise, treat a missing account ID as a blocker.
- Treat a missing relevant error code as a blocker when the
  issue is an error-related technical problem.
- Follow the required output schema.

{format_instructions}
"""
    ),
    (
        "user",
        """
Structured support summary:
{support_summary}
"""
    )
]).partial(
    format_instructions=ex5_step2_parser.get_format_instructions()
)

In [11]:
ex5_step2_chain = (
    ex5_step2_prompt
    | llm
    | ex5_step2_parser
)

In [12]:
escalation_readiness = ex5_step2_chain.invoke({
    "support_summary": support_summary.model_dump_json(
        indent=2
    )
})

print(escalation_readiness.model_dump_json(indent=2))

{
  "missing_fields": [
    "account ID"
  ],
  "is_ready_to_escalate": false
}


## Step 3 — Template: ChatPromptTemplate (system persona: "support operations lead")  |  Parser: StrOutputParser



In [13]:
ex5_step3_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a support operations lead preparing an
escalation report.

Use only the supplied structured support summary and
escalation-readiness review.

Include:
1. Issue summary
2. Customer tone
3. Troubleshooting already attempted
4. Missing information
5. Escalation readiness
6. Recommended next action
7. Provisional priority with a brief rationale

Rules:
- Do not invent customer details or technical facts.
- Do not claim that an issue has been resolved unless
  the supplied data confirms it.
- If escalation is blocked, state what must be collected
  before proceeding under the stated escalation rules.
- If the case is ready, explain the next escalation step.
- Base the priority on the reported impact and urgency.
- If impact is unclear, state that priority needs confirmation.
- Keep the report concise and professional.
"""
    ),
    (
        "user",
        """
STRUCTURED SUPPORT SUMMARY:
{support_summary}

ESCALATION READINESS REVIEW:
{escalation_readiness}

Prepare the escalation report for the support lead.
"""
    )
])

In [14]:
ex5_step3_chain = (
    ex5_step3_prompt
    | llm
    | StrOutputParser()
)

In [15]:
escalation_report = ex5_step3_chain.invoke({
    "support_summary": support_summary.model_dump_json(
        indent=2
    ),
    "escalation_readiness":
        escalation_readiness.model_dump_json(indent=2)
})

print(escalation_report)

**Escalation Report – Support Lead**

| Section | Content |
|---------|---------|
| **1. Issue Summary** | Customer encountered an **Error 502** during the checkout process. |
| **2. Customer Tone** | Frustrated. |
| **3. Troubleshooting Already Attempted** | • Customer refreshed the page.<br>• Customer logged out and logged back in.<br>• Support suggested a page refresh and requested the account ID. |
| **4. Missing Information** | *Account ID* – required to locate the customer’s session and transaction logs. |
| **5. Escalation Readiness** | **Not ready** – escalation blocked until the missing account ID is obtained. |
| **6. Recommended Next Action** | 1. Prompt the customer for their account ID (or provide a secure method to share it).<br>2. Once received, gather any relevant logs or screenshots of the 502 error.<br>3. Proceed with escalation to the engineering team once all required data is collected. |
| **7. Provisional Priority** | **High** – checkout failure directly impacts r

In [16]:
def process_support_escalation(chat_log_text: str):

    # Step 1: Normalize the raw support chat.
    support_summary = ex5_step1_chain.invoke({
        "chat_log_text": chat_log_text
    })

    # Step 2: Evaluate escalation readiness.
    escalation_readiness = ex5_step2_chain.invoke({
        "support_summary":
            support_summary.model_dump_json(indent=2)
    })

    # Step 3: Generate the escalation report.
    escalation_report = ex5_step3_chain.invoke({
        "support_summary":
            support_summary.model_dump_json(indent=2),
        "escalation_readiness":
            escalation_readiness.model_dump_json(indent=2)
    })

    return {
        "support_summary": support_summary,
        "escalation_readiness": escalation_readiness,
        "escalation_report": escalation_report
    }

In [17]:
result = process_support_escalation(chat_log_text)

print("STEP 1: SUPPORT SUMMARY")
print(
    result["support_summary"].model_dump_json(indent=2)
)

print("\nSTEP 2: ESCALATION READINESS")
print(
    result["escalation_readiness"].model_dump_json(
        indent=2
    )
)

print("\nSTEP 3: ESCALATION REPORT")
print(result["escalation_report"])

STEP 1: SUPPORT SUMMARY
{
  "issue": "Error 502 encountered during checkout",
  "customer_tone": "Frustrated",
  "resolution_attempted": "Customer refreshed the page and logged out and back in; support suggested refreshing and asked for account ID"
}

STEP 2: ESCALATION READINESS
{
  "missing_fields": [
    "Account ID"
  ],
  "is_ready_to_escalate": false
}

STEP 3: ESCALATION REPORT
**Escalation Report – Support Lead**

| Section | Content |
|---------|---------|
| **1. Issue Summary** | Customer encountered an **Error 502** during the checkout process. |
| **2. Customer Tone** | Frustrated. |
| **3. Troubleshooting Already Attempted** | • Customer refreshed the page.<br>• Customer logged out and logged back in.<br>• Support suggested a page refresh and requested the account ID. |
| **4. Missing Information** | *Account ID* – required to locate the customer’s session and transaction logs. |
| **5. Escalation Readiness** | **Not ready** – escalation blocked until the missing account I